In [9]:
import numpy as np 
import pandas as pd 
import geopandas as gpd
import utils.cross_validation as cval
import torch
from torch import nn
import matplotlib.pyplot as plt


In [2]:
#Neural network
class MLP(nn.Module):
   
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential( #Passes through layers sequentially
             nn.Linear(17, 42, bias= True), #100 inputs into 200 perceptrons
             nn.ReLU(), #Activation function
             nn.Linear(42, 50, bias= True), # 200 inputs 100 outputs
             nn.ReLU(),
             nn.Linear(50, 2, bias= True), # 200 inputs 100 outputs
             #nn.Dropout(0.05)
         )

    def forward(self, x): #Feed input X into layers mentioned above
      return self.layers(x)
    
#The dataloader, tensor conversions etc
class Dataset(torch.utils.data.Dataset): #Initialsing a structure to pass each line of data through

    def __init__(self, X, y, scale_data=True):
      if not torch.is_tensor(X) and not torch.is_tensor(y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

In [82]:


N_EPOCHS = 5 #config['training']['epochs']
TEST_SIZE = 0.4 # config['data']['test_size']
BATCH_SIZE = 32 #config['training']['batch_size']
# DATA_PATH = config['data']['path']

df= pd.read_csv("data/final/fd_df.csv")
PID_loc= pd.read_csv("data/lookup/PID_location_all.csv")

ecoregions=cval.process_ecoregion("data/Ecoregions/Ecoregions2017.shp")



In [83]:
df=df.merge(PID_loc, on="PID", how="left")
df = df.drop(columns=[col for col in df.columns if "_y" in col])
df.columns = df.columns.str.replace('_x$', '', regex=True)



In [84]:
def assign_spatial_groups(df, grid_size=1.0):
    df = df.copy()

    print('fixed')
    df["lon_bin"] = (df["lon"] // grid_size) * grid_size
    df["lat_bin"] = (df["lat"] // grid_size) * grid_size
    
    df["spatial_group"] = (
        df["biome"].astype(str) + "_" +
        df["lon_bin"].astype(str) + "_" +
        df["lat_bin"].astype(str)
    )
    
    return df

In [85]:
df=assign_spatial_groups(df, grid_size=0.005)

fixed


In [86]:
df['spatial_group'].value_counts()

spatial_group
6_-121.61_44.07                  5719719
6_-121.69500000000001_46.95      3836475
6_-123.845_41.61                 3252117
6_-124.405_48.1                  3207600
6_-123.875_41.935                2913597
                                  ...   
6_-123.62_40.1                        27
6_-123.16_39.775                      25
3_-119.45_37.14                       25
3_-122.635_38.800000000000004         16
6_-123.795_46.03                       9
Name: count, Length: 3197, dtype: int64

In [87]:
df.dropna(subset=["lat", "lon"], inplace=True)



gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"  # WGS84
)

In [88]:

grouped_df = gpd.sjoin(
    gdf,
    ecoregions,
    how="left",          # keep all PIDs
    predicate="within"   # point inside polygon
)

grouped_df.drop_duplicates(inplace=True)

KeyboardInterrupt: 

In [ ]:
grouped_df['ECO_ID'].value_counts()

,PID,Raos_Q,Functional_Evenness,Mean Pairwise D,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km,SG_SOC_Content_015cm,EsaCci_BurntAreasProbability,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE
0,1_16_17_122_1,2.753350,0.885053,4.008434,57.464227,1414.712036,37.898327,53.300079,46.942509,0.0,...,NE05,2,361,56.924527,35.905513,Nature Could Reach Half Protected,#ACC13E,#458970,#7BC141,CC-BY 4.0
3969,1_16_17_122_1,2.753350,0.885053,4.008434,57.464227,1414.712036,37.898327,53.300079,46.942509,0.0,...,NE05,2,361,56.924527,35.905513,Nature Could Reach Half Protected,#ACC13E,#458970,#7BC141,CC-BY 4.0
7938,1_16_17_122_1,2.753350,0.885053,4.008434,57.464227,1414.712036,37.898327,53.300079,46.942509,0.0,...,NE05,2,361,56.924527,35.905513,Nature Could Reach Half Protected,#ACC13E,#458970,#7BC141,CC-BY 4.0
11907,1_16_17_122_1,2.753350,0.885053,4.008434,57.464227,1414.712036,37.898327,53.300079,46.942509,0.0,...,NE05,2,361,56.924527,35.905513,Nature Could Reach Half Protected,#ACC13E,#458970,#7BC141,CC-BY 4.0
15876,1_16_17_122_1,2.753350,0.885053,4.008434,57.464227,1414.712036,37.898327,53.300079,46.942509,0.0,...,NE05,2,361,56.924527,35.905513,Nature Could Reach Half Protected,#ACC13E,#458970,#7BC141,CC-BY 4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7885121,1_16_49_519_1,1.602024,0.740934,2.340108,44.386789,1363.343018,29.499975,41.027515,38.046040,0.0,...,NE05,3,367,126.922931,20.091093,Nature Could Recover,#058047,#458970,#F9A91B,CC-BY 4.0
7886210,1_16_49_519_1,1.602024,0.740934,2.340108,44.386789,1363.343018,29.499975,41.027515,38.046040,0.0,...,NE05,3,367,126.922931,20.091093,Nature Could Recover,#058047,#458970,#F9A91B,CC-BY 4.0
7887299,1_16_49_519_1,1.602024,0.740934,2.340108,44.386789,1363.343018,29.499975,41.027515,38.046040,0.0,...,NE05,3,367,126.922931,20.091093,Nature Could Recover,#058047,#458970,#F9A91B,CC-BY 4.0
7888388,1_16_49_519_1,1.602024,0.740934,2.340108,44.386789,1363.343018,29.499975,41.027515,38.046040,0.0,...,NE05,3,367,126.922931,20.091093,Nature Could Recover,#058047,#458970,#F9A91B,CC-BY 4.0


In [80]:
grouped_df['ECO_ID'].value_counts()

ECO_ID
361    1033
367     374
Name: count, dtype: int64

In [76]:
def ecoregion_cross_validation(gdf, ecoregion, test_size, batch_size):

    grouped_df = gpd.sjoin(
        gdf,
        ecoregion,
        how="left",          # keep all PIDs
        predicate="within"   # point inside polygon
    )

    grouped_df.dropna(subset=['ECO_ID'], inplace=True)

    # unique groups
    groups = grouped_df["ECO_ID"].unique()
    grouped_df = grouped_df.groupby("spatial_group").filter(lambda x: len(x) >= batch_size)
    print(len(groups))
    # number to sample
    n_select = int(len(groups) * test_size)


    selected_groups = np.random.choice(groups, size=n_select, replace=False)
    test = grouped_df[grouped_df["spatial_group"].isin(selected_groups)]
    train = grouped_df[~grouped_df["spatial_group"].isin(selected_groups)]
    
    return train, test

In [77]:
train, test= ecoregion_cross_validation(gdf, ecoregions, TEST_SIZE, BATCH_SIZE)

# train = train.loc[:, :'csc']
# test = test.loc[:, :'csc']

# y=['transformed npp', 'csc'] 

# X_train=train.drop(columns=y+['PID']).values
# y_train = np.column_stack([train['transformed npp'], train['csc']])

# X_test=test.drop(columns=y+['PID']).values
# y_test = np.column_stack([test['transformed npp'], test['csc']])

# dataset = Dataset(X_train, y_train)
# testset= Dataset(X_test, y_test)

2


In [78]:
test

,PID,Raos_Q,Functional_Evenness,Mean Pairwise D,CHELSA_BIO_Annual_Mean_Temperature,CHELSA_BIO_Annual_Precipitation,CHELSA_BIO_Precipitation_Seasonality,CrowtherLab_SoilMoisture_intraAnnualSD_downsampled10km,SG_SOC_Content_015cm,EsaCci_BurntAreasProbability,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE


In [49]:
#Saving results
results = {} 
vloss=[]
tloss=[]
testloss=[]
rv=[]
rt=[]
lrs = []

#Initialize the NN
model = MLP()
criterion = nn.L1Loss() #Loss function
optimizer = torch.optim.SGD(model.parameters(), lr=0.001) #Gradient descent
# lambda1 = lambda epoch: 0.65 ** epoch
# scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda1)

trainloader = torch.utils.data.DataLoader( dataset,  batch_size=BATCH_SIZE, num_workers=8)
valloader = torch.utils.data.DataLoader( dataset, batch_size=BATCH_SIZE, num_workers=8)
testloader = torch.utils.data.DataLoader( testset, batch_size=BATCH_SIZE, num_workers=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)


In [52]:
print(len(trainloader))

40721


In [50]:
for epoch in range(N_EPOCHS): 
    train_loss = 0.0
    valid_loss = 0.0

    #train the model -------------------------------
    model.train() # prep model for training

    for data, target in trainloader:
        data=data.float()
        target=target.float()
        # target = target.reshape((target.shape[0], 1))

        output= model(data.float()) #Feedforward
        optimizer.zero_grad()
        loss = criterion(output, target)

        loss.backward() #backward pass: compute gradient of the loss with respect to model parameters
        optimizer.step() #perform a single optimization step (parameter update)
        lrs.append(optimizer.param_groups[0]["lr"])
        # scheduler.step()
        train_loss += loss.item()
    
    print('finish training')

KeyboardInterrupt: 